# Partie A — Exploration, nettoyage et baseline

ArSarcasm-v2. Chaque étape est dans le notebook : chargement, audit du texte brut, split, nettoyage, TF-IDF, entraînement, puis évaluation sur le test officiel.

Le fichier de test n'ajuste pas le TF-IDF, n'entraîne pas le classifieur et ne choisit pas le seuil. Il est noté seulement à la fin, avec le modèle déjà fixé.

In [1]:
from pathlib import Path
import re
import time

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from pyarabic.araby import DIACRITICS, TATWEEL
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_rows", 80)

SEED = 42
np.random.seed(SEED)

plt.rcParams["font.family"] = "Segoe UI"


## 1. Chargement

Les deux CSV officiels sont lus depuis GitHub, puis gardés dans `data/raw/` pour les lancements suivants. `sarcasm` devient un booléen. L'identifiant est posé avant tout split.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
CACHE = ROOT / "data" / "raw"
CACHE.mkdir(parents=True, exist_ok=True)


def read_split(name):
    path = CACHE / name
    if not path.exists():
        downloaded = pd.read_csv(BASE_URL + name)
        downloaded.to_csv(path, index=False)
    frame = pd.read_csv(path)
    frame["tweet"] = frame["tweet"].fillna("").astype(str)
    frame["sarcasm"] = (
        frame["sarcasm"].astype(str).str.strip().str.upper().isin(["TRUE", "1", "T", "YES"])
    )
    frame["sentiment"] = frame["sentiment"].astype(str)
    frame["dialect"] = frame["dialect"].astype(str)
    return frame


full = read_split("training_data.csv")
test = read_split("testing_data.csv")
full["id"] = range(len(full))
test["id"] = range(len(test))

print(f"fichier train officiel : {len(full)}")
print(f"fichier test officiel  : {len(test)}")
print("colonnes :", list(full.columns))

## 2. Aperçu du tableau, puis des textes bruts

D'abord l'en-tête des deux fichiers. Ensuite les premiers tweets tels qu'ils sont stockés, avant tout nettoyage : guillemets d'export, liens, `@`, `#`, emojis, tashkil, tatweel.

In [ ]:
print("En-tête du fichier d'entraînement")
display(full.head())
print("En-tête du fichier de test")
display(test.head())
print("Premiers textes bruts")
for tweet in full["tweet"].head(8):
    print("---")
    print(tweet)

## 3. Audit du texte brut

Comptage des indices de surface sur le fichier d'entraînement, avant le split et avant le nettoyage. La barre donne la part des tweets qui portent chaque indice.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"(?<!\w)@\w+")
HASHTAG_RE = re.compile(r"(?<!\w)#\w+")
LAUGH_RE = re.compile(r"ه{2,}")
REPEATED_PUNCT_RE = re.compile(r"([!?؟.,،؛…])\1+")
REPEATED_LETTER_RE = re.compile(r"([^\W\d_])\1{2,}")
LATIN_RE = re.compile(r"[A-Za-z]")
INDIAN_DIGIT_RE = re.compile(r"[٠-٩]")
WS_RE = re.compile(r"\s+")
DIACRITIC_CHARS = "".join(ch for ch in DIACRITICS if ch != TATWEEL)
DIACRITIC_RE = re.compile("[" + re.escape(DIACRITIC_CHARS) + "]")
TATWEEL_RE = re.compile(re.escape(TATWEEL))


def has_csv_quote_wrap(text):
    # Guillemets ASCII autour de tout le tweet : artefact de l'export CSV.
    stripped = str(text).strip()
    return len(stripped) >= 2 and stripped[0] == '"' and stripped[-1] == '"'


def has_emoji(text):
    return len(emoji.emoji_lis(str(text))) > 0


CUES = [
    ("url", lambda text: URL_RE.search(text) is not None),
    ("mention", lambda text: MENTION_RE.search(text) is not None),
    ("hashtag", lambda text: HASHTAG_RE.search(text) is not None),
    ("emoji", has_emoji),
    ("laugh_heh", lambda text: LAUGH_RE.search(text) is not None),
    ("repeated_punctuation", lambda text: REPEATED_PUNCT_RE.search(text) is not None),
    ("diacritics", lambda text: DIACRITIC_RE.search(text) is not None),
    ("tatweel", lambda text: TATWEEL_RE.search(text) is not None),
    ("repeated_letters", lambda text: REPEATED_LETTER_RE.search(text) is not None),
    ("latin", lambda text: LATIN_RE.search(text) is not None),
    ("indian_digits", lambda text: INDIAN_DIGIT_RE.search(text) is not None),
    ("csv_quote_wrap", has_csv_quote_wrap),
]

tweets = full["tweet"].astype(str)
labels = full["sarcasm"].astype(bool)
rows = []
for name, present_fn in CUES:
    present = tweets.map(present_fn).astype(bool)
    present_labels = labels[present]
    absent_labels = labels[~present]
    correlation = present.astype(float).corr(labels.astype(float))
    rows.append(
        {
            "feature": name,
            "n_tweets": int(present.sum()),
            "frequency": float(present.mean()),
            "sarcasm_rate_present": float(present_labels.mean()) if present.any() else float("nan"),
            "sarcasm_rate_absent": float(absent_labels.mean()) if (~present).any() else float("nan"),
            "pearson_with_sarcasm": float(correlation) if pd.notna(correlation) else float("nan"),
        }
    )

audit = pd.DataFrame(rows)
display(audit)

fig, ax = plt.subplots(figsize=(8, 5))
order = audit.sort_values("frequency")
ax.barh(order["feature"], order["frequency"])
ax.set_xlabel("part des tweets")
ax.set_title("Indices de surface — fichier d'entraînement brut")
fig.tight_layout()
display(fig)
plt.close(fig)

## 4. Split imposé et distribution

10 % de validation, graine 42, stratifié sur `sarcasm`. Les graphes portent sur train et val. Le tableau des taux inclut aussi le test officiel, pour le compte-rendu seulement.

In [ ]:
train, val = train_test_split(
    full,
    test_size=0.10,
    random_state=SEED,
    stratify=full["sarcasm"],
)
train = train.reset_index(drop=True).copy()
val = val.reset_index(drop=True).copy()
assert set(train["id"]).isdisjoint(set(val["id"]))
print(f"train : {len(train)}")
print(f"val   : {len(val)}")


def sarcasm_rate_table(splits, by=None):
    frames = []
    for name, frame in splits.items():
        y = frame["sarcasm"].astype(bool)
        if by is None:
            frames.append(
                pd.DataFrame(
                    {
                        "split": [name],
                        "n": [len(frame)],
                        "n_sarcastic": [int(y.sum())],
                        "sarcasm_rate": [float(y.mean()) if len(frame) else float("nan")],
                    }
                )
            )
            continue
        grouped = frame.assign(_label=y).groupby(by, dropna=False)["_label"]
        part = grouped.agg(n="size", n_sarcastic="sum", sarcasm_rate="mean").reset_index()
        part.insert(0, "split", name)
        frames.append(part)
    return pd.concat(frames, ignore_index=True)


rates = sarcasm_rate_table({"train": train, "val": val, "test": test})
display(rates)
display(sarcasm_rate_table({"train": train, "val": val, "test": test}, by="dialect"))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_names = ["train", "val"]
plot_frames = [train, val]
counts = [int(frame["sarcasm"].sum()) for frame in plot_frames]
non_counts = [len(frame) - n for frame, n in zip(plot_frames, counts)]
x = np.arange(len(plot_names))
axes[0].bar(x - 0.18, non_counts, width=0.36, label="non sarcastique")
axes[0].bar(x + 0.18, counts, width=0.36, label="sarcastique")
axes[0].set_xticks(x, plot_names)
axes[0].set_ylabel("tweets")
axes[0].set_title("Effectifs")
axes[0].legend()
rates_plot = [n / len(frame) for n, frame in zip(counts, plot_frames)]
axes[1].bar(plot_names, rates_plot)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("taux de sarcasme")
axes[1].set_title("Taux")
fig.tight_layout()
display(fig)
plt.close(fig)


def pies(splits, column, title, order):
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4.2))
    for ax, (name, frame) in zip(axes, splits.items()):
        counts = frame[column].astype(str).value_counts().reindex(order).fillna(0)
        counts = counts[counts > 0]
        total = float(counts.sum())
        wedges, _texts = ax.pie(counts.to_numpy(), startangle=90, counterclock=False)
        labels = [f"{label} — {100 * float(counts[label]) / total:.1f}%" for label in counts.index]
        ax.legend(wedges, labels, loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=False)
        ax.set_title(f"{name}\nn={len(frame)}")
    fig.suptitle(title)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


pies(
    {"train": train, "val": val},
    "sentiment",
    "Sentiment (train et val)",
    ["NEG", "NEU", "POS"],
)
pies(
    {"train": train, "val": val},
    "dialect",
    "Dialecte (train et val)",
    ["msa", "egypt", "gulf", "levant", "magreb"],
)

## 5. Vides et doublons

Trois cas dans le train, avant le graphe :

- **Copies en trop** : le même texte est répété. On garde une ligne et on retire les autres.
- **Groupes aux deux labels** : le même tweet est répété, mais une copie est sarcastique et l'autre non. Les labels se contredisent, donc on retire tout le groupe.
- **Fuites test** : un texte du train est déjà dans le fichier de test officiel. On le retire du train pour ne pas apprendre sur un texte du test.

La validation n'est pas modifiée, pour garder le split de 10 %.

In [ ]:
def strip_csv_quotes(text):
    stripped = str(text).strip()
    if len(stripped) >= 2 and stripped[0] == '"' and stripped[-1] == '"':
        return stripped[1:-1].strip()
    return stripped


def normalize_for_dedup(text):
    # Clé de comparaison : sans guillemets d'export ni tatweel.
    cleaned = strip_csv_quotes(text).replace(TATWEEL, "")
    return WS_RE.sub(" ", cleaned).strip()


empty_mask = train["tweet"].map(lambda text: normalize_for_dedup(text) == "")
extra_mask = train["tweet"].duplicated(keep="first")
keys = train["tweet"].map(normalize_for_dedup)
nunique = train.assign(_key=keys).groupby("_key")["sarcasm"].nunique()
bad_keys = set(nunique[nunique > 1].index)
contradictory_mask = keys.isin(bad_keys)
test_exact = set(test["tweet"].astype(str))
test_norm = set(test["tweet"].map(normalize_for_dedup))
train_norm = keys
leak_mask = train["tweet"].astype(str).isin(test_exact) | train_norm.isin(test_norm)
val_norm = val["tweet"].map(normalize_for_dedup)
n_val_overlap = int((val["tweet"].astype(str).isin(test_exact) | val_norm.isin(test_norm)).sum())

print("Textes vides : plus aucun caractère une fois les guillemets et le tatweel retirés.")
print(f"nombre : {int(empty_mask.sum())}")
display(train.loc[empty_mask, ["id", "tweet", "sarcasm"]].head(10))

print("Copies en trop : le même texte est répété. On garde la première ligne.")
print(f"nombre : {int(extra_mask.sum())}")
exact_groups = train["tweet"].duplicated(keep=False)
display(train.loc[exact_groups, ["id", "tweet", "sarcasm"]].sort_values("tweet").head(12))

print("Groupes aux deux labels : le même tweet, une fois sarcastique et une fois non. On retire tout le groupe.")
print(f"groupes : {len(bad_keys)} — lignes : {int(contradictory_mask.sum())}")
display(train.loc[contradictory_mask, ["id", "tweet", "sarcasm"]].sort_values("tweet").head(12))

print("Fuites test : ce texte du train est déjà dans le test officiel. On le retire du train seulement.")
print(f"nombre : {int(leak_mask.sum())}")
display(train.loc[leak_mask, ["id", "tweet", "sarcasm"]])
print(f"Même chevauchement dans la validation, laissé en place : {n_val_overlap}")
print("Le graphe reprend ces quatre comptes.")

fig, ax = plt.subplots(figsize=(7, 4))
labels_bar = ["vides", "copies en trop", "labels contradictoires", "fuites test"]
values_bar = [
    int(empty_mask.sum()),
    int(extra_mask.sum()),
    int(contradictory_mask.sum()),
    int(leak_mask.sum()),
]
ax.bar(labels_bar, values_bar)
ax.set_ylabel("lignes du train")
ax.set_title("Lignes à retirer du train")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
n_before = len(train)
# Une copie exacte au même label est gardée. Un texte aux deux labels est retiré en entier.
drop_mask = empty_mask | (extra_mask & ~contradictory_mask) | contradictory_mask | leak_mask
train = train.loc[~drop_mask].reset_index(drop=True).copy()
print(f"train avant : {n_before}")
print(f"train après : {len(train)}")
print(f"lignes retirées : {int(drop_mask.sum())}")
print(f"val inchangée : {len(val)}")
print("taux après nettoyage du train")
display(sarcasm_rate_table({"train": train, "val": val, "test": test}))
display(sarcasm_rate_table({"train": train, "val": val, "test": test}, by="dialect"))

## 6. Liens, mentions et hashtags

Nouvelle colonne `text`. Les URL et les `@mentions` sont retirés. Le `#` est retiré, le mot du hashtag reste. Les guillemets d'export sont retirés. La même fonction est appliquée à train, val et test.

In [ ]:
def strip_special(text):
    cleaned = strip_csv_quotes(text)
    cleaned = URL_RE.sub(" ", cleaned)
    cleaned = MENTION_RE.sub(" ", cleaned)
    cleaned = re.sub(r"(?<!\w)#(\w+)", r"\1", cleaned)
    return WS_RE.sub(" ", cleaned).strip()


preview = train[["tweet"]].copy()
preview["text"] = preview["tweet"].map(strip_special)
display(preview.loc[preview["tweet"] != preview["text"]].head(6))

for frame in (train, val, test):
    frame["text"] = frame["tweet"].map(strip_special)

## 7. Correspondance des emojis

Chaque emoji distinct reçoit un nom arabe une seule fois. Le nom vient des mots de son libellé, traduits dans l'ordre. Ensuite ce nom remplace l'emoji dans le texte, pour que le TF-IDF le voie comme des mots.

In [ ]:
from collections import Counter

EMOJI_AR = {
    '2': '٢',
    'a': '',
    'A': 'أ',
    'Africa': 'أفريقيا',
    'alarm': 'منبه',
    'Americas': 'الأمريكتان',
    'and': 'و',
    'anger': 'غضب',
    'angry': 'غاضب',
    'anguished': 'متألم',
    'anxious': 'قلق',
    'apple': 'تفاحة',
    'Arab': 'عربي',
    'Arabia': 'العربية',
    'arrow': 'سهم',
    'arrows': 'أسهم',
    'articulated': 'مفصلي',
    'astonished': 'مندهش',
    'automobile': 'سيارة',
    'baby': 'طفل',
    'back': 'خلف',
    'backhand': 'ظهر اليد',
    'bag': 'حقيبة',
    'bags': 'حقائب',
    'ball': 'كرة',
    'balloon': 'بالون',
    'bandage': 'ضمادة',
    'bank': 'بنك',
    'beaming': 'مشرق',
    'beating': 'خافق',
    'bell': 'جرس',
    'bellhop': 'حامل حقائب',
    'beverage': 'مشروب',
    'biceps': 'عضلة',
    'big': 'كبير',
    'birthday': 'عيد ميلاد',
    'black': 'أسود',
    'blond': 'أشقر',
    'blood': 'دم',
    'blossom': 'زهرة',
    'blowing': 'ينفخ',
    'blue': 'أزرق',
    'board': 'لوحة',
    'bomb': 'قنبلة',
    'book': 'كتاب',
    'books': 'كتب',
    'bottle': 'زجاجة',
    'bouquet': 'باقة',
    'bowing': 'ينحني',
    'boy': 'ولد',
    'broken': 'مكسور',
    'but': 'لكن',
    'butterfly': 'فراشة',
    'button': 'زر',
    'cake': 'كعكة',
    'camera': 'كاميرا',
    'car': 'سيارة',
    'cat': 'قط',
    'check': 'صح',
    'cherry': 'كرز',
    'chick': 'كتكوت',
    'Christmas': 'عيد الميلاد',
    'circle': 'دائرة',
    'circled': 'محاط',
    'clapper': 'كلاكيت',
    'clapping': 'تصفيق',
    'Claus': 'كلوز',
    'clock': 'ساعة',
    'clockwise': 'مع عقارب الساعة',
    'closed': 'مغلق',
    'cloud': 'غيمة',
    'clover': 'برسيم',
    'clown': 'مهرج',
    'collision': 'اصطدام',
    'confetti': 'قصاصات',
    'confounded': 'مرتبك',
    'confused': 'حائر',
    'cork': 'سدادة',
    'couple': 'زوج',
    'crescent': 'هلال',
    'cross': 'صليب',
    'crossbones': 'عظام متقاطعة',
    'crown': 'تاج',
    'crying': 'يبكي',
    'crystal': 'كريستال',
    'curly': 'مجعد',
    'cyclone': 'إعصار',
    'dancing': 'يرقص',
    'dark': 'داكن',
    'decoration': 'زخرفة',
    'diamond': 'معين',
    'disappointed': 'خائب',
    'dizzy': 'دائر',
    'dog': 'كلب',
    'double': 'مزدوج',
    'dove': 'حمامة',
    'down': 'أسفل',
    'downcast': 'مغموم',
    'droplets': 'قطرات',
    'drops': 'قطرات',
    'eggplant': 'باذنجان',
    'Egypt': 'مصر',
    'eight': 'ثمانية',
    'eighteen': 'ثمانية عشر',
    'Emirates': 'الإمارات',
    'entry': 'دخول',
    'envelope': 'ظرف',
    'Europe': 'أوروبا',
    'evil': 'شرير',
    'ewe': 'نعجة',
    'exclamation': 'تعجب',
    'exploding': 'ينفجر',
    'expressionless': 'بلا تعبير',
    'eyes': 'عيون',
    'face': 'وجه',
    'facing': 'مقابل',
    'fallen': 'ساقط',
    'fear': 'خوف',
    'fearful': 'خائف',
    'finger': 'إصبع',
    'fire': 'نار',
    'fist': 'قبضة',
    'flag': 'علم',
    'flash': 'وميض',
    'flexed': 'مثنية',
    'floor': 'أرض',
    'flushed': 'محمر',
    'fluttering': 'يرفرف',
    'folded': 'متشابكة',
    'food': 'طعام',
    'four': 'أربعة',
    'frog': 'ضفدع',
    'from': 'من',
    'front': 'أمام',
    'frowning': 'عابس',
    'full': 'بدر',
    'gem': 'جوهرة',
    'gesturing': 'يومئ',
    'gift': 'هدية',
    'girl': 'بنت',
    'glasses': 'نظارات',
    'globe': 'كرة أرضية',
    'glowing': 'متوهج',
    'green': 'أخضر',
    'grimacing': 'متجهم',
    'grinning': 'مبتسم',
    'growing': 'ينمو',
    'hair': 'شعر',
    'hand': 'يد',
    'hands': 'يدان',
    'hat': 'قبعة',
    'head': 'رأس',
    'headphone': 'سماعة',
    'hear': 'يسمع',
    'heart': 'قلب',
    'hearts': 'قلوب',
    'herb': 'عشبة',
    'hibiscus': 'كركديه',
    'high': 'عال',
    'hole': 'حفرة',
    'horns': 'قرون',
    'horse': 'حصان',
    'hot': 'حار',
    'hugging': 'يعانق',
    'hundred': 'مئة',
    'hushed': 'صامت',
    'in': 'في',
    'incoming': 'قادم',
    'index': 'سبابة',
    'jack': 'جاك',
    'joy': 'فرح',
    'key': 'مفتاح',
    'keyboard': 'لوحة مفاتيح',
    'keycap': 'مفتاح رقم',
    'kiss': 'قبلة',
    'kissing': 'يقبل',
    'Kuwait': 'الكويت',
    'label': 'ملصق',
    'lantern': 'فانوس',
    'laptop': 'حاسوب',
    'large': 'كبير',
    'Latvia': 'لاتفيا',
    'laughing': 'يضحك',
    'leaf': 'ورقة',
    'Lebanon': 'لبنان',
    'left': 'يسار',
    'letter': 'رسالة',
    'Libya': 'ليبيا',
    'light': 'فاتح',
    'lightning': 'برق',
    'locked': 'مقفل',
    'loop': 'حلقة',
    'lorry': 'شاحنة',
    'loudly': 'بصوت عال',
    'love': 'حب',
    'M': 'م',
    'man': 'رجل',
    'mark': 'علامة',
    'mask': 'كمامة',
    'medal': 'ميدالية',
    'medical': 'طبي',
    'medium': 'متوسط',
    'megaphone': 'مكبر صوت',
    'meridians': 'خطوط طول',
    'Mexico': 'المكسيك',
    'microphone': 'ميكروفون',
    'middle': 'وسطى',
    'military': 'عسكري',
    'moai': 'مواي',
    'mobile': 'جوال',
    'money': 'مال',
    'monkey': 'قرد',
    'moon': 'قمر',
    'mouth': 'فم',
    'movie': 'فيلم',
    'musical': 'موسيقي',
    'nail': 'ظفر',
    'national': 'وطني',
    'nauseated': 'متقزز',
    'nerd': 'مثقف',
    'neutral': 'محايد',
    'new': 'جديد',
    'newspaper': 'جريدة',
    'nib': 'ريشة',
    'nine': 'تسعة',
    'no': 'لا',
    'nose': 'أنف',
    'note': 'ملاحظة',
    'notebook': 'دفتر',
    'notepad': 'مفكرة',
    'notes': 'نوتات',
    'o': 'و',
    "o'clock": 'تمام',
    'of': 'من',
    'ogre': 'غول',
    'OK': 'موافق',
    'on': 'على',
    'oncoming': 'مقبل',
    'one': 'واحد',
    'open': 'مفتوح',
    'orange': 'برتقالي',
    'over': 'فوق',
    'package': 'طرد',
    'Palestinian': 'فلسطيني',
    'palm': 'كف',
    'palms': 'كفان',
    'panda': 'باندا',
    'park': 'حديقة',
    'party': 'حفلة',
    'partying': 'يحتفل',
    'pensive': 'متأمل',
    'persevering': 'مثابر',
    'person': 'شخص',
    'phone': 'هاتف',
    'pile': 'كومة',
    'pinching': 'يقرص',
    'pistol': 'مسدس',
    'pleading': 'متوسل',
    'pointed': 'مسنن',
    'pointing': 'يشير',
    'points': 'نقاط',
    'police': 'شرطة',
    'polish': 'طلاء',
    'poo': 'براز',
    'popper': 'مفرقعات',
    'popping': 'تنفتح',
    'pouting': 'عابس',
    'princess': 'أميرة',
    'prohibited': 'ممنوع',
    'purple': 'بنفسجي',
    'pushpin': 'دبوس',
    'question': 'سؤال',
    'radio': 'راديو',
    'rain': 'مطر',
    'raised': 'مرفوع',
    'raising': 'يرفع',
    'recycling': 'إعادة تدوير',
    'red': 'أحمر',
    'relieved': 'مرتاح',
    'revolving': 'دوار',
    'ribbon': 'شريط',
    'rice': 'أرز',
    'right': 'يمين',
    'ring': 'خاتم',
    'robot': 'روبوت',
    'rocket': 'صاروخ',
    'rolled': 'ملفوف',
    'rolling': 'يدور',
    'rose': 'وردة',
    'round': 'مستدير',
    'running': 'يركض',
    'sad': 'حزين',
    'Santa': 'بابا نويل',
    'Saudi': 'السعودية',
    'savoring': 'يتذوق',
    'score': 'نتيجة',
    'screaming': 'يصرخ',
    'see': 'يرى',
    'seedling': 'شتلة',
    'shamrock': 'برسيم',
    'sheaf': 'حزمة',
    'shopping': 'تسوق',
    'showing': 'يظهر',
    'shushing': 'يسكت',
    'sign': 'لافتة',
    'skin': 'بشرة',
    'skull': 'جمجمة',
    'sleeping': 'نائم',
    'sleepy': 'نعسان',
    'slightly': 'قليلا',
    'small': 'صغير',
    'smiling': 'مبتسم',
    'smirking': 'بابتسامة ماكرة',
    'snow': 'ثلج',
    'snowflake': 'ندفة ثلج',
    'snowman': 'رجل ثلج',
    'soccer': 'كرة قدم',
    'Spain': 'إسبانيا',
    'sparkle': 'بريق',
    'sparkles': 'بريق',
    'sparkling': 'لامع',
    'speak': 'يتكلم',
    'spiral': 'لولب',
    'square': 'مربع',
    'squinting': 'يضيق عينيه',
    'star': 'نجمة',
    'States': 'الولايات',
    'steam': 'بخار',
    'stone': 'حجر',
    'suit': 'مجموعة',
    'sun': 'شمس',
    'sunflower': 'دوار الشمس',
    'sunglasses': 'نظارات شمسية',
    'sweat': 'عرق',
    'symbol': 'رمز',
    'tear': 'دمعة',
    'tears': 'دموع',
    'technologist': 'تقني',
    'telephone': 'هاتف',
    'television': 'تلفاز',
    'tennis': 'تنس',
    'Territories': 'الأراضي',
    'the': '',
    'thinking': 'يفكر',
    'thirty': 'ثلاثون',
    'thought': 'فكرة',
    'thumbs': 'إبهام',
    'tired': 'متعب',
    'together': 'معا',
    'tone': 'لون',
    'tongue': 'لسان',
    'top': 'علوي',
    'trade': 'تجارة',
    'tree': 'شجرة',
    'triangle': 'مثلث',
    'triangular': 'مثلثي',
    'tulip': 'توليب',
    'turban': 'عمامة',
    'Turkey': 'تركيا',
    'twelve': 'اثنا عشر',
    'two': 'اثنان',
    'type': 'فصيلة',
    'umbrella': 'مظلة',
    'unamused': 'غير مستمتع',
    'under': 'تحت',
    'United': 'المتحدة',
    'unlocked': 'مفتوح',
    'up': 'أعلى',
    'upside': 'مقلوب',
    'vertical': 'عمودي',
    'victory': 'نصر',
    'video': 'فيديو',
    'voltage': 'جهد',
    'walking': 'يمشي',
    'water': 'ماء',
    'waving': 'يلوح',
    'wearing': 'يرتدي',
    'weary': 'منهك',
    'white': 'أبيض',
    'wind': 'ريح',
    'winking': 'يغمز',
    'with': 'مع',
    'without': 'بدون',
    'woman': 'امرأة',
    'worried': 'قلق',
    'wrapped': 'مغلف',
    'writing': 'كتابة',
    'yellow': 'أصفر',
    'zany': 'مجنون',
    'zzz': 'نوم',
    'Bahrain': 'البحرين',
    'Italy': 'إيطاليا',
    'Oman': 'عمان',
    'Qatar': 'قطر',
    'beer': 'جعة',
    'behind': 'خلف',
    'bird': 'طائر',
    'boat': 'قارب',
    'brown': 'بني',
    'bulb': 'مصباح',
    'cane': 'عكاز',
    'cherries': 'كرز',
    'clinking': 'قرع',
    'cold': 'بارد',
    'counterclockwise': 'عكس عقارب الساعة',
    'crossed': 'متقاطع',
    'deaf': 'أصم',
    'disguised': 'متنكر',
    'downwards': 'للأسفل',
    'drooling': 'يسيل لعابه',
    'drop': 'قطرة',
    'droplet': 'قطرة',
    'duck': 'بطة',
    'facepalming': 'كف على الوجه',
    'female': 'أنثى',
    'fingers': 'أصابع',
    'flower': 'زهرة',
    'getting': 'يصير',
    'goblin': 'عفريت',
    'grapes': 'عنب',
    'halo': 'هالة',
    'handshake': 'مصافحة',
    'knocked': 'مصدوم',
    'link': 'رابط',
    'mailbox': 'صندوق بريد',
    'male': 'ذكر',
    'mango': 'مانجو',
    'maple': 'قيقب',
    'massage': 'تدليك',
    'melon': 'شمام',
    'mugs': 'كؤوس',
    'otter': 'قضاعة',
    'out': 'خارج',
    'pear': 'كمثرى',
    'rainbow': 'قوس قزح',
    'registered': 'مسجل',
    'repeat': 'تكرار',
    'rowing': 'تجديف',
    'shrugging': 'يهز كتفيه',
    'snake': 'ثعبان',
    'speech': 'كلام',
    'standing': 'واقف',
    'strawberry': 'فراولة',
    'struck': 'مصعوق',
    'syringe': 'حقنة',
    'violin': 'كمان',
    'warning': 'تحذير',
    'watch': 'ساعة يد',
    'wilted': 'ذابلة',
}


def emoji_symbols(text):
    return [hit["emoji"] for hit in emoji.emoji_lis(str(text).replace("\ufe0f", ""))]


def arabic_emoji_name(symbol):
    # Appelé une fois par emoji distinct, pas une fois par occurrence.
    english = emoji.demojize(symbol).strip(":")
    parts = english.replace("-", " ").replace("_", " ").replace("’", "'").split()
    words = []
    for part in parts:
        token = part.strip("()")
        words.append(EMOJI_AR.get(token, EMOJI_AR.get(token.lower(), token)))
    return " ".join(word for word in words if word).strip()


found = Counter()
for text in train["text"]:
    found.update(emoji_symbols(text))

symbols = set(found)
for frame in (val, test):
    for text in frame["text"]:
        symbols.update(emoji_symbols(text))

arabic_by_emoji = {symbol: arabic_emoji_name(symbol) for symbol in symbols}
print(f"emojis distincts, un nom chacun : {len(arabic_by_emoji)}")

emoji_map = pd.DataFrame(
    {
        "emoji": list(found.keys()),
        "arabe": [arabic_by_emoji[symbol] for symbol in found.keys()],
        "n_train": list(found.values()),
    }
).sort_values("n_train", ascending=False)
display(emoji_map.head(20))

top = emoji_map.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["arabe"], top["n_train"])
ax.set_xlabel("occurrences dans le train")
ax.set_title("Emojis les plus fréquents")
fig.tight_layout()
display(fig)
plt.close(fig)


def map_emojis(text):
    value = str(text).replace("\ufe0f", "")
    for symbol in set(emoji_symbols(value)):
        value = value.replace(symbol, " " + arabic_by_emoji[symbol] + " ")
    return WS_RE.sub(" ", value).strip()


preview = train.loc[train["text"].map(lambda text: len(emoji_symbols(text)) > 0), ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(map_emojis)
display(preview)

for frame in (train, val, test):
    frame["text"] = frame["text"].map(map_emojis)


## 8. Tashkil, tatweel et lettres répétées

Retrait des diacritiques arabes et du tatweel. Une suite de trois lettres identiques ou plus est ramenée à une seule lettre.

In [ ]:
def strip_arabic_noise(text):
    cleaned = DIACRITIC_RE.sub("", str(text))
    cleaned = cleaned.replace(TATWEEL, "")
    cleaned = re.sub(r"([^\W\d_])\1{2,}", r"\1", cleaned)
    return WS_RE.sub(" ", cleaned).strip()


noise = train["text"].map(
    lambda text: DIACRITIC_RE.search(text) is not None
    or TATWEEL in text
    or REPEATED_LETTER_RE.search(text) is not None
)
preview = train.loc[noise, ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(strip_arabic_noise)
display(preview)

for frame in (train, val, test):
    frame["text"] = frame["text"].map(strip_arabic_noise)

print(f"textes vides après nettoyage — train : {int((train['text'] == '').sum())}")
print(f"textes vides après nettoyage — val   : {int((val['text'] == '').sum())}")

## 9. Mots latins et chiffres

Chaque mot en lettres latines est réécrit lettre par lettre en arabe. Chaque chiffre occidental (0–9) devient un chiffre arabe (٠–٩). La même table est appliquée à train, val et test.

In [ ]:
LATIN_LETTER_AR = {
    "a": "ا", "b": "ب", "c": "ك", "d": "د", "e": "ي",
    "f": "ف", "g": "ج", "h": "ه", "i": "ي", "j": "ج",
    "k": "ك", "l": "ل", "m": "م", "n": "ن", "o": "و",
    "p": "ب", "q": "ق", "r": "ر", "s": "س", "t": "ت",
    "u": "و", "v": "ف", "w": "و", "x": "كس", "y": "ي", "z": "ز",
}
DIGIT_AR = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
letter_table = pd.DataFrame({"latin": list(LATIN_LETTER_AR), "arabe": list(LATIN_LETTER_AR.values())})
digit_table = pd.DataFrame({"latin": list("0123456789"), "arabe": list("٠١٢٣٤٥٦٧٨٩")})
display(letter_table)
display(digit_table)


def latin_token_to_arabic(token):
    pieces = []
    for char in token:
        if char.isdigit():
            pieces.append(char.translate(DIGIT_AR))
        else:
            pieces.append(LATIN_LETTER_AR.get(char.lower(), char))
    return "".join(pieces)


def map_latin(text):
    return re.sub(r"[A-Za-z0-9]+", lambda match: latin_token_to_arabic(match.group(0)), str(text))


has_latin = train["text"].map(lambda text: re.search(r"[A-Za-z0-9]", text) is not None)
preview = train.loc[has_latin, ["text"]].head(6).copy()
preview["apres"] = preview["text"].map(map_latin)
display(preview)

for frame in (train, val, test):
    frame["text"] = frame["text"].map(map_latin)


## 10. TF-IDF

Mots (1–2) et caractères (2–5). Le vectoriseur est ajusté sur `train["text"]` seulement.

In [ ]:
word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
char_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 5))
tfidf = FeatureUnion([("word", word_vectorizer), ("char", char_vectorizer)])
X_train = tfidf.fit_transform(train["text"])

print(f"documents train : {X_train.shape[0]}")
print(f"traits          : {X_train.shape[1]}")
print(f"vocabulaire mots        : {len(word_vectorizer.vocabulary_)}")
print(f"vocabulaire caractères  : {len(char_vectorizer.vocabulary_)}")

## 11. Entraînement

### Classifieur trivial « jamais sarcastique »

L'accuracy est élevée parce que la classe sarcastique est minoritaire. La métrique retenue est le F1 de cette classe.

In [ ]:
y_val = val["sarcasm"].astype(bool).to_numpy()
trivial_pred = np.zeros(len(val), dtype=bool)
trivial_proba = np.zeros(len(val), dtype=float)
trivial_acc = accuracy_score(y_val, trivial_pred)
trivial_f1 = f1_score(y_val, trivial_pred, pos_label=True, zero_division=0)
print(f"accuracy val : {trivial_acc:.4f}")
print(f"F1 sarcastique val : {trivial_f1:.4f}")

### Q1 — à reporter dans le compte-rendu

Recopier les taux train / val / test affichés après le nettoyage des lignes, globaux et par dialecte, ainsi que l'accuracy et le F1 du classifieur trivial. La métrique retenue est le F1 de la classe sarcastique : l'accuracy du classifieur trivial est forte alors que son F1 sarcastique est nul.

### Baseline TF-IDF + régression logistique

`class_weight="balanced"`. Le seuil est le plus petit seuil de la grille qui maximise le F1 sarcastique sur la validation.

In [ ]:
X_val = tfidf.transform(val["text"])
clf = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=SEED,
)
started = time.perf_counter()
clf.fit(X_train, train["sarcasm"].astype(bool))
elapsed = time.perf_counter() - started
n_params = int(clf.coef_.size + clf.intercept_.size)

classes = list(clf.classes_)
sarcasm_index = classes.index(True)
val_proba = clf.predict_proba(X_val)[:, sarcasm_index]

grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
best_threshold = float(grid[0])
best_f1 = -1.0
for threshold in grid:
    f1 = f1_score(y_val, val_proba >= threshold, pos_label=True, zero_division=0)
    if f1 > best_f1:
        best_f1 = float(f1)
        best_threshold = float(threshold)
val_pred = val_proba >= best_threshold

print(f"seuil : {best_threshold:.2f}")
print(f"F1 sarcastique val : {best_f1:.4f}")
print(f"temps d'entraînement : {elapsed:.1f} s")
print(f"paramètres entraînés : {n_params}")

### Matrice de confusion

Vérité en ligne, prédiction en colonne, sur la validation et au seuil choisi.

In [ ]:
cm = confusion_matrix(y_val, val_pred, labels=[False, True])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm,
    display_labels=["non sarcastique", "sarcastique"],
).plot(ax=ax, colorbar=False)
ax.set_title("Matrice de confusion — validation")
fig.tight_layout()
display(fig)
plt.close(fig)


### Raccourcis à une seule règle

URL, mention et guillemets d'export, lus sur le tweet d'origine : ces indices ne sont plus dans `text`. La direction de la règle vient du taux de sarcasme du train. Le F1 est calculé sur la validation.

In [ ]:
def add_rule_features(frame):
    out = frame.copy()
    raw = out["tweet"].astype(str)
    out["has_url"] = raw.map(lambda text: URL_RE.search(text) is not None)
    out["has_mention"] = raw.map(lambda text: MENTION_RE.search(text) is not None)
    out["has_csv_quote"] = raw.map(has_csv_quote_wrap)
    return out


train_rules = add_rule_features(train)
val_rules = add_rule_features(val)
y_train = train_rules["sarcasm"].astype(bool)
rule_summary = []
rule_predictions = []
for name, column in (
    ("rule_url", "has_url"),
    ("rule_mention", "has_mention"),
    ("rule_csv_quote", "has_csv_quote"),
):
    present_train = train_rules[column].astype(bool)
    rate_present = float(y_train[present_train].mean()) if present_train.any() else float("nan")
    rate_absent = float(y_train[~present_train].mean()) if (~present_train).any() else float("nan")
    positive_when_present = bool(rate_present >= rate_absent)
    present_val = val_rules[column].astype(bool).to_numpy()
    pred = present_val if positive_when_present else ~present_val
    proba = pred.astype(float)
    rule_summary.append(
        {
            "model": name,
            "feature": column,
            "positive_when_present": positive_when_present,
            "train_rate_present": rate_present,
            "train_rate_absent": rate_absent,
            "val_f1_sarcasm": float(f1_score(y_val, pred, pos_label=True, zero_division=0)),
        }
    )
    rule_predictions.append((name, pred, proba))

rules = pd.DataFrame(rule_summary)
display(rules)
print(
    "Chaque règle est un classifieur à un seul indice de surface. "
    "Comparer son F1 au F1 trivial et au F1 de la baseline dit si l'indice suffit "
    "à approcher le score du modèle. Le seuil et les règles n'ont pas vu le test."
)

f1_names = ["trivial", "url", "mention", "csv_quote", "tfidf"]
f1_values = [trivial_f1] + rules["val_f1_sarcasm"].tolist() + [best_f1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(f1_names, f1_values)
ax.set_ylim(0, 1)
ax.set_ylabel("F1 sarcastique sur val")
ax.set_title("Trivial, règles de surface, baseline")
fig.tight_layout()
display(fig)
plt.close(fig)

### Q2 — à reporter dans le compte-rendu

Recopier la fréquence de l'artefact de guillemets, de l'URL et des mentions (audit du texte brut), leur corrélation avec le label, et le F1 de chaque règle sur la validation. Conclure sur ce que ces raccourcis autorisent à dire des scores.

## 12. Test officiel

Le modèle est celui entraîné sur le train. Le seuil est celui choisi sur la validation. Le test officiel n'a servi ni à l'ajustement du TF-IDF, ni à l'entraînement, ni au choix du seuil. On y mesure l'accuracy et la matrice de confusion.

In [ ]:
y_test = test["sarcasm"].astype(bool).to_numpy()
X_test = tfidf.transform(test["text"])
test_proba = clf.predict_proba(X_test)[:, sarcasm_index]
test_pred = test_proba >= best_threshold

val_acc = accuracy_score(y_val, val_pred)
test_acc = accuracy_score(y_test, test_pred)
print(f"seuil fixé sur la validation : {best_threshold:.2f}")
print(f"accuracy validation : {val_acc:.4f}")
print(f"accuracy test       : {test_acc:.4f}")

cm_test = confusion_matrix(y_test, test_pred, labels=[False, True])
print("non sarcastique bien classé :", int(cm_test[0, 0]))
print("non sarcastique prédit sarcastique :", int(cm_test[0, 1]))
print("sarcastique prédit non sarcastique :", int(cm_test[1, 0]))
print("sarcastique bien classé :", int(cm_test[1, 1]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm_test,
    display_labels=["non sarcastique", "sarcastique"],
).plot(ax=ax, colorbar=False)
ax.set_title("Matrice de confusion — test officiel")
fig.tight_layout()
display(fig)
plt.close(fig)


## Journal

Les colonnes de test restent vides. Relancer cette cellule remplace les lignes de la partie A et laisse les autres modèles en place.

In [ ]:
def metric_row(model, preprocess, y_pred, proba, threshold, train_time_s, n_trainable, notes, include_accuracy=False):
    return {
        "model": model,
        "preprocess": preprocess,
        "seed": SEED,
        "val_f1_sarcasm": round(float(f1_score(y_val, y_pred, pos_label=True, zero_division=0)), 6),
        "val_pr_auc": round(float(average_precision_score(y_val.astype(int), proba)), 6),
        "val_macro_f1": round(float(f1_score(y_val, y_pred, average="macro", zero_division=0)), 6),
        "val_accuracy": round(float(accuracy_score(y_val, y_pred)), 6) if include_accuracy else "",
        "test_f1_sarcasm": "",
        "test_pr_auc": "",
        "test_macro_f1": "",
        "threshold": "" if threshold is None else round(float(threshold), 2),
        "train_time_s": round(float(train_time_s), 3),
        "n_trainable_params": int(n_trainable),
        "notes": notes,
    }


rows = [
    metric_row(
        "trivial_never_sarcastic",
        "none",
        trivial_pred,
        trivial_proba,
        None,
        0.0,
        0,
        "toujours non sarcastique ; l'accuracy n'est pas la métrique de sélection",
        include_accuracy=True,
    ),
    metric_row(
        "tfidf_word12_char25_logreg",
        "clean_special_emoji_tashkeel",
        val_pred,
        val_proba,
        best_threshold,
        elapsed,
        n_params,
        "seuil choisi sur val par F1 sarcastique maximal ; test non utilisé",
    ),
]
for name, pred, proba in rule_predictions:
    rows.append(
        metric_row(
            name,
            "raw",
            pred,
            proba,
            0.5,
            0.0,
            0,
            "direction choisie sur le taux de sarcasme du train",
        )
    )

columns = list(rows[0].keys())
fresh = pd.DataFrame(rows, columns=columns)
path = ROOT / "results" / "results.csv"
if path.exists() and path.stat().st_size > 0:
    previous = pd.read_csv(path)
    if "model" in previous.columns and not previous.empty:
        previous = previous.loc[~previous["model"].isin(fresh["model"])]
        fresh = pd.concat([previous, fresh], ignore_index=True)
fresh = fresh.reindex(columns=columns)
path.parent.mkdir(parents=True, exist_ok=True)
fresh.to_csv(path, index=False)
display(fresh)
print("Le fichier de prédictions de test n'est pas écrit. L'accuracy et la matrice sont dans la section 12.")